# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook is a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset on human protein abundance and modifications, via the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json), using the `mlcroissant` library.

### Dataset Source
The dataset is described and accessed using the Croissant schema URL above.

In [ ]:
# Ensure mlcroissant is installed and up-to-date
!pip install --upgrade mlcroissant

## 1. Data Loading
Load metadata and records using `mlcroissant`. This step initializes the dataset using the Croissant schema URL and inspects key metadata fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata summary
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published Date:", getattr(metadata, 'datePublished', 'N/A'))
print("Croissant Version:", getattr(metadata, 'conformsTo', 'N/A'))
print("License:", getattr(metadata, 'license', 'N/A'))
print("Available Record Sets (@id):")
recordset_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
for rid in recordset_ids:
    print("  -", rid)


## 2. Data Overview
Explore available record sets and fields defined in the Croissant schema. Each record set and field/column is referenced by its Croissant `@id` attribute.

In [ ]:
# We'll print out key attributes of each record set: fields, columns, data types, etc.

# The Croissant metadata uses @id references for entities.
from typing import List

def get_recordsets(metadata) -> List[dict]:
    """Return the list of record sets from the metadata object."""
    jsondata = metadata.to_json()
    return jsondata.get('recordSet', [])

def get_fields_from_recordset(recordset: dict):
    # Each record set may list 'field' as a field or 'column' if CSV/TSV
    fields = recordset.get('field', [])
    # Normalize to always a list
    if isinstance(fields, dict):
        fields = [fields]
    columns = recordset.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    return fields, columns

recordsets = get_recordsets(dataset.metadata)
if not recordsets:
    print("No record sets listed directly in the root metadata (the inlined schema may reference them in FileObjects instead).\nListing DataFileObjects as candidate tabular files...")
    datafile_objs = [dist for dist in dataset.metadata.to_json().get('distribution', []) if dist.get('@type', '') == 'DataFileObject']
    if datafile_objs:
        for fobj in datafile_objs:
            print(f"- {fobj['@id']}: {fobj.get('name', fobj.get('contentUrl', 'N/A'))}")
    else:
        print("No tabular DataFileObjects in schema. Can't show record sets.")
else:
    for recordset in recordsets:
        rid = recordset["@id"]
        name = recordset.get('name', 'N/A')
        fields, columns = get_fields_from_recordset(recordset)
        print(f"\nRecordSet @id: {rid}")
        print(f"  Name: {name}")
        if fields:
            print("  Fields (field @id):")
            for f in fields:
                print("    -", f["@id"], ":", f.get('name','N/A'))
        if columns:
            print("  Columns (column @id):")
            for c in columns:
                print("    -", c["@id"], ":", c.get('name','N/A'))
# If record sets are not in the 'recordSet' root, hint the user where to find them.

## 3. Data Extraction
Load records from one or more record sets into pandas DataFrames. All references to record sets and fields must use Croissant `@id` attributes.

If no record set is declared in the root schema (as in this FAIR² schema), try to find tabular resources in 'distribution'.

In [ ]:
#--- Find record sets or tabular objects to load ---#
metadata_json = dataset.metadata.to_json()

# The schema may embed tabular files in distribution/DataFileObject.
# Let's try to infer record_set_id.
if metadata_json.get('recordSet'):
    record_sets_to_load = [rs['@id'] for rs in metadata_json['recordSet']]
else:
    # Sometimes record sets are defined in distribution
    # For this schema (2024 versions), the main table is likely in distribution[0]
    # as the main data file with extraced protein info.
    record_sets_to_load = []
    distributions = metadata_json.get('distribution', [])
    for dist in distributions:
        # Identify DataFileObject or files with tabular data
        if 'DataFileObject' in dist.get('@type', ''):
            record_sets_to_load.append(dist['@id'])
        else:
            # Fallback: just use the distribution @id
            record_sets_to_load.append(dist['@id'])

print("Extracting data from record set(s):")
print(record_sets_to_load)

# Load each as a DataFrame
dataframes = {}
for record_set_id in record_sets_to_load:
    try:
        records = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Choose a record set to display
if record_sets_to_load:
    main_record_set_id = record_sets_to_load[0]
    if main_record_set_id in dataframes:
        print(f"Columns in DataFrame for {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
        display(dataframes[main_record_set_id].head())
    else:
        print(f"No DataFrame loaded for {main_record_set_id}.")
else:
    print("No record sets or distributions found to extract.")

## 4. Exploratory Data Analysis (EDA)

In this section, we'll demonstrate basic filtering, normalization, and grouping of fields/columns. Please ensure you use the correct field `@id` to access DataFrame columns. Adapt the field IDs based on what you observed above.

In [ ]:
# For demonstration, pick a numeric field (e.g., 'MW_kDa' if exists, or any numeric column)

# Use DataFrame from the first record set
record_set_id = main_record_set_id
df = dataframes.get(record_set_id)
if df is not None and not df.empty:
    # Try to suggest likely numeric fields
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try some typical known fields from proteomics datasets
        for try_col in ['MW_kDa', 'molecular_weight', 'coverage_percent', 'unique_peptides']:
            if try_col in df.columns:
                numeric_candidates.append(try_col)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use the first
        print(f"Using numeric field: {numeric_field}")
    else:
        print("No numeric field found in the data.\nAvailable columns:", df.columns.tolist())
        numeric_field = None

    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].dtype in [float, int] else 10  # default if can't compute
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Add a normalized field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Use a group field (for proteins, often 'accession', 'gene_symbol', or experiment/sample)
        group_field_candidates = [c for c in df.columns if 'sample' in c.lower() or 'group' in c.lower() or 'accession' in c.lower()]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No appropriate group field found.")
else:
    print("No DataFrame available for analysis.")

## 5. Visualization
Create a simple plot to visualize the distribution of values in a key numeric field, or relationships between two variables. Requires matplotlib and seaborn for advanced visuals.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping is available
    if len(group_field_candidates) > 0:
        # Show boxplot of the numeric value by group
        plt.figure(figsize=(12, 6))
        top_groups = df[group_field_candidates[0]].value_counts().index[:5]
        sns.boxplot(x=group_field_candidates[0], y=numeric_field, data=df[df[group_field_candidates[0]].isin(top_groups)])
        plt.title(f'{numeric_field} by {group_field_candidates[0]} (Top 5 Groups)')
        plt.show()
else:
    print("No numeric field or data for plotting.")

## 6. Conclusion
- We successfully loaded the FAIR² dataset using the Croissant schema and `mlcroissant`.
- Available record sets, fields, and columns have been enumerated by Croissant `@id`.
- A tabular record set was loaded, with filtering, normalization, and grouping demonstrated on a numeric field (by `@id`).
- Visualizations highlighted data distributions and group-wise summaries.

The Croissant schema's use of globally unique `@id` references ensures robust, FAIR-compliant tabular data workflows. For in-depth use, see [`mlcroissant` documentation](https://github.com/mlcommons/croissant) and schema details at the [dataset source URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).